In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import pyterrier_deepct
import pyterrier as pt
if not pt.started():
    pt.init() 
path = '/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/'
deepct = pyterrier_deepct.DeepCTTransformer(path + "bert_files/bert-base-uncased/bert_config.json", path + "DeepCT_train_output/topics_conclusion/model.ckpt-153596", vocab_file = path + "bert_files/bert-base-uncased/vocab.txt", max_seq_length=512)

In [ ]:
from tqdm import tqdm
import ir_datasets
dataset = ir_datasets.load('argsme/2020-04-01')
splitted_df = pd.DataFrame()
ids = []
splitted_text = []
for doc in tqdm(dataset.docs_iter(), total=387740):
    ids.append(doc.doc_id)
    splitted_text.append(doc.premises_texts)

In [ ]:
splitted_df['docno'] = ids
splitted_df['text'] = splitted_text
splitted_df

In [ ]:
with tf.device('/GPU:0'):
    transformed = deepct.transform(splitted_df)

In [ ]:
print('yeap?')

In [ ]:
from xml.etree import ElementTree
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics2020.xml')
data = tree.getroot()
titles = []
title_ids = []
nums = []
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    nums.append(num)
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics.xml')
data = tree.getroot()    
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    nums.append(num)
topics = pd.DataFrame()
topics['qid'] = nums
topics['query'] = titles

qrels_path=("/workspace/src/evaluate_targers/qrels.qrels")
qrels = pt.io.read_qrels(qrels_path)

docnos = qrels.docno

In [ ]:
save_path = '/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/'
# df_test = pd.read_csv(save_path + 'topics_conclusions_transformed.csv')

path_targer = '/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/targer_indexes/'
pd_indexer = pt.DFIndexer(path_targer + "pd_index_deepct-test-no-split")
indexref = pd_indexer.index(transformed["text"], transformed["docno"])
# indexref = pt.IndexRef.of("/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/targer_indexes/pd_index_deepct-conclusions1/data.properties")

index = pt.IndexFactory.of(indexref)
retr = pt.BatchRetrieve(index, controls = {"wmodel": "TF_IDF"})
res=retr.transform(topics)

res

In [ ]:
eval = pt.Utils.evaluate(res,qrels,metrics=["ndcg_cut_25", "ndcg_cut_5", "bpref"])
eval

In [ ]:
res_copy = res.copy()
res_copy = res_copy[res_copy['docno'].isin(docnos)]
res_copy.head(10)

eval = pt.Utils.evaluate(res_copy,qrels,metrics=["ndcg_cut_5", "bpref", "ndcg_cut_25"])
eval

In [ ]:
transformed.to_csv(path_or_buf = '/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/test-no-split-transformed.csv')

In [ ]:
transformed.head()

In [ ]:
splitted_df.head()

In [ ]:
transformed['text'][2]

In [ ]:
splitted_df['text'][2]

In [ ]:
for tag_text in transformed['text']:
    len_tag.append(len(tokenizer.tokenize(tag_text)))

lens = np.array([len_tag])
print(f'Mean of tagged lenghtes in tokens: {np.mean(len_tag)}')

print(f'Variance of tagged lenghtes in tokens: {np.var(len_tag)}')

print(f'Standard deviation of tagged lenghtes in tokens: {np.std(len_tag)}')

print(f'Median of tagged lenghtes in tokens: {np.median(len_tag)}')

print(f'Max of tagged lenghtes in tokens: {np.max(len_tag)}')

print(f'Min of tagged lenghtes in tokens: {np.min(len_tag)}')
